# Guide06: Final Test, Report, and Deployment

Last of six notebooks on one running example: predicting Ames house prices. This one covers **Stages 10-11** of the [13-stage workflow](../Guide00_Supervised-ML_Linear_Regression_end-to-end_workflow.md), with a look at 12-13: open the locked test set for the first and only time, report the result honestly, and ship the whole pipeline.

**Prerequisite:** `Guide05_Supervised-ML_Linear-Regression_Cross-Validation_Regularization_GridSearch.ipynb`.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 15)
plt.rcParams["figure.figsize"] = (7, 4)


---
## Recap: The Frozen Configuration

`Guide05` froze one configuration: **Lasso, `alpha=0.0003`, on `Guide03`'s feature set** (no polynomial terms). Rebuild it exactly, fit on training data only -- `X_test` and `y_test` have not been opened by any of the five notebooks before this one.


In [ ]:
import warnings

from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

from pipeline.ames_workflow import load_ames, clean_ames, split_ames, add_engineered_features, build_preprocessor

X_train, X_test, y_train, y_test = split_ames(clean_ames(load_ames()))
X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)
print(f"train: {X_train.shape}   test: {X_test.shape}  (about to be opened, for the first time, below)")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    frozen_model = TransformedTargetRegressor(
        regressor=Pipeline([("prep", build_preprocessor(X_train)), ("model", Lasso(alpha=0.0003, max_iter=30000))]),
        func=np.log, inverse_func=np.exp,
    ).fit(X_train, y_train)


**Write the expectation down before looking.** `Guide05` reported a cross-validated MAE of about \$13,580, and a more honest, nested-CV estimate of about \$13,920. Expect the test score to land somewhere near the higher of the two, plausibly a bit above it -- cross-validation's own best score is known to be mildly optimistic.


---
## Stage 10: Final Test

One prediction pass over `X_test`, once:


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

test_preds = frozen_model.predict(X_test)
test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_r2 = r2_score(y_test, test_preds)

print(f"FINAL TEST (2010, {len(X_test)} sales): MAE=${test_mae:,.0f}   RMSE=${test_rmse:,.0f}   R2={test_r2:.3f}")
print(f"for comparison -- Guide05's CV estimate: ~$13,580   nested estimate: ~$13,920")


The test score, about \$15,100, comes in higher than either cross-validation number -- worse than the optimistic \$13,580, and still a bit worse than the more honest \$13,920. $R^2$ is still a respectable 0.925, so this is not a broken model; it is a model that generalizes a little less well than training-time estimates suggested. **This number is not tunable.** Nothing about `frozen_model` changes because of what we just saw -- reacting to it would turn this single look into another round of tuning, exactly what Rule 2 warns against. What follows is diagnosis for the report, not a search for a fix.

### Why might the gap exist?

A few honest, partial explanations, not one single cause:


In [ ]:
print("mean price -- train (2006-2009): $", round(y_train.mean()), " test (2010): $", round(y_test.mean()))
for col in ["GrLivArea", "OverallQual", "HouseAge", "TotalSF"]:
    print(f"{col:12s} train mean={X_train[col].mean():8.1f}   test mean={X_test[col].mean():8.1f}")


Feature means barely move between the two periods -- this is not a case of the test houses being obviously different kinds of houses. Two smaller, real effects likely combine instead: 2010 sits at the tail of the housing-market crash `Guide01` flagged, when prices had started easing in ways four earlier years of training data could not fully anticipate; and with only 164 test sales, some of the gap is simply the noise of a small sample. Neither explanation licenses a change to the model -- they explain the number, they do not excuse improving on it after the fact.


---
## Residuals, for the Report

### The worst predictions


In [ ]:
worst = (
    pd.DataFrame({"actual": y_test, "predicted": test_preds, "abs_error": np.abs(y_test - test_preds)})
    .join(X_test[["Neighborhood", "SaleCondition", "OverallQual", "GrLivArea"]])
    .sort_values("abs_error", ascending=False)
    .head(5)
)
worst


In [ ]:
worst10_conditions = (
    pd.DataFrame({"abs_error": np.abs(y_test - test_preds)})
    .join(X_test[["SaleCondition"]])
    .sort_values("abs_error", ascending=False)
    .head(10)["SaleCondition"]
    .value_counts()
)
print("SaleCondition among the worst 10 (test):", worst10_conditions.to_dict())
print("SaleCondition among the worst 10 (train, from Guide03): {'Partial': 7, 'Normal': 2, 'Family': 1}")


A different picture from `Guide03`'s training-set finding: there, `Partial` sales dominated the worst errors; here, mostly ordinary `Normal` sales do. With only 164 test rows this could easily be noise rather than a real reversal -- but it is also possible that the model's `Partial`-sale weakness turned out to matter less on this particular future year than it did during training, simply because `Partial` sales are a smaller share of 2010 (4.3%) than of 2006-2009 (9.4%). Report the finding as it actually came out, not as the one `Guide03` primed us to expect.

### Error by price band


In [ ]:
price_band = pd.qcut(y_test, 4)
band_errors = pd.DataFrame({
    "abs_error": np.abs(y_test - test_preds),
    "pct_error": np.abs(y_test - test_preds) / y_test * 100,
}).groupby(price_band, observed=True).mean()
band_errors.round(1)


Dollar error grows with price, as `Guide03` already found on training data; percentage error stays in a roughly similar range across bands, which is what the log-target transform was for.


---
## Model Card

**Question.** How much will a house in Ames, Iowa sell for, given its characteristics? (`Guide01`, Stage 1.)

**Data.** 1,377 arms-length residential sales in Ames, Iowa, 2006-2010, after removing two documented new-construction sales recorded before completion (`Guide01`, Stage 3). Trained on 2006-2009 (1,213 sales); tested on 2010 alone (164 sales), never touched before this notebook.

**Method.** Linear regression on a log-transformed target, Lasso-regularized (`alpha=0.0003`), over imputed/encoded/scaled features: numeric columns, ten ordinal quality grades, and one-hot-encoded categoricals with rare levels pooled. Polynomial and interaction terms were tried (`Guide04`) and did not improve on this configuration.

**Performance.** MAE about \$15,100, $R^2$ about 0.93 on the 2010 test set -- the honest number to quote, not the more optimistic \$13,580 cross-validation estimate from training data.

**Most influential features** (Lasso, log-price scale; direction and rough size, not a substitute for the full coefficient table): `Neighborhood` (especially `Crawfor`, `StoneBr`), `GrLivArea`, `OverallQual`, and total square footage. Coefficient size is not a reliable importance ranking on its own here -- features are on different scales, several are correlated, and Lasso's penalty shrinks and sometimes zeroes coefficients for reasons that mix "genuinely unimportant" with "redundant with another surviving feature."

**Limitations.** Reflects Ames, Iowa in 2006-2010; the 2010 test result suggests performance a bit below the training-time estimate on genuinely new sales. Errors grow in dollar terms for expensive houses. No claim of causal effects: this model was built for prediction, not inference (`Guide00`, Section 6).

**Reproduce it.** `pipeline/ames_workflow.py`, `Guide01`-`Guide06`, `random_state=42` throughout, scikit-learn 1.1+.


---
## Stage 11: Deploy

### Retrain on all available data

The frozen configuration is not changing; the *data* it trains on grows once there is no more test set to protect. Refit the same configuration on every row, train and test combined:


In [ ]:
X_all = pd.concat([X_train, X_test])
y_all = pd.concat([y_train, y_test])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    deployed_model = TransformedTargetRegressor(
        regressor=Pipeline([("prep", build_preprocessor(X_all)), ("model", Lasso(alpha=0.0003, max_iter=30000))]),
        func=np.log, inverse_func=np.exp,
    ).fit(X_all, y_all)

print(f"deployed_model fitted on all {len(X_all)} rows (2006-2010)")


### Save the whole pipeline, not just the model

`deployed_model` already contains preprocessing, the target transform, and the model as one object -- saving it is one line, and it is the *entire* thing that must travel together:


In [ ]:
import joblib

joblib.dump(deployed_model, "ames_model.joblib")

reloaded_model = joblib.load("ames_model.joblib")
sample = X_all.iloc[[0]]
same_prediction = np.allclose(deployed_model.predict(sample), reloaded_model.predict(sample))
print(f"reloaded model gives the same prediction: {same_prediction}")
print(f"example prediction: ${reloaded_model.predict(sample)[0]:,.0f}")


### Validate incoming data before it reaches the model

New data will not always look like training data. Three checks, on the reloaded model:


In [ ]:
# 1. A required column is missing entirely.
missing_col = sample.drop(columns=["GrLivArea"])
try:
    reloaded_model.predict(missing_col)
except ValueError as e:
    print("missing column ->", str(e)[:80])


In [ ]:
# 2. A brand-new, never-seen NOMINAL category (e.g. a data-entry typo in Neighborhood).
new_neighborhood = sample.copy()
new_neighborhood["Neighborhood"] = "Mordor"
pred = reloaded_model.predict(new_neighborhood)
print(f"unseen nominal category -> no error, predicted ${pred[0]:,.0f} "
      f"(handled by min_frequency's 'infrequent' bucket from Guide02)")


In [ ]:
# 3. A brand-new, never-seen ORDINAL category (e.g. a typo in a quality grade).
new_quality = sample.copy()
new_quality["KitchenQual"] = "Legendary"
try:
    reloaded_model.predict(new_quality)
except ValueError as e:
    print("unseen ordinal category ->", str(e)[:80])


Two very different failure behaviors from two encoders built in `Guide02`, and it matters which is which before this ships anywhere: a missing column or an unrecognized quality grade stops the pipeline outright, loudly, which is the safe failure for something the model has no sound way to interpret. An unfamiliar neighborhood name is absorbed into the "infrequent" bucket instead of erroring, which is convenient, but worth knowing about -- a real new neighborhood would get whatever prediction that bucket implies, silently, not a warning that something unusual came in. A production system should still validate the incoming columns and value ranges itself, rather than relying on the pipeline's own errors as its only line of defense.


---
## Stages 12-13: A Preview of Monitoring and Feedback

Both are properly beyond what a notebook does, but this guide series already practiced a piece of each without naming it that way.

**Monitoring:** the entire final test just performed *is* a first drift check, not only an accuracy check -- 2010 is a genuinely future year no training or tuning decision ever saw, and the gap between the cross-validated estimate and the test score is itself a measurement of how much performance can slip once real time moves forward. In production, that check would repeat continuously: watch whether incoming feature distributions keep resembling `X_train`'s (`Guide00`'s "data drift"), and watch prediction error once true sale prices become known, the same comparison this notebook just made once.

**Feedback:** `Guide00`'s Stage 13 table says where a problem sends you back to. The test-set gap here traces mainly to a data problem (a shifting market, not a code or modeling error) -- which would send a real project back to Stage 2/3, gathering more recent sales, rather than back to Stage 9 to retune a configuration that was not the cause.


---
## Series Wrap-Up

Six notebooks, one example, thirteen stages:

1. `Guide01` defined the problem, understood, cleaned, split, and explored the data.
2. `Guide02` transformed it into a matrix a linear model could use, without leaking.
3. `Guide03` built two baselines and diagnosed them honestly.
4. `Guide04` tried more flexibility, and found its cost before finding its benefit.
5. `Guide05` controlled that flexibility with cross-validation and regularization.
6. `Guide06` looked at the test set once, reported the result plainly, and shipped the whole pipeline.

The two rules from `Guide00` held throughout: every transformation was fit on training data only, and the test set was touched exactly once, at the very end, for a number that was reported rather than chased.

Next: `01_Classification/`, where the same thirteen stages apply with a categorical target.


---
## Your Turn: A Leakage Audit

The snippet below is a sketch, not something to run as-is (`X`, `y` stand in for a numeric feature matrix and target you already have in memory). It has **five** planted mistakes, each one this guide series specifically warned about. Find and fix all five by inspection, the way you would review a colleague's notebook -- none of the five would raise an error or a warning if you ran a completed version. That is the point: leakage does not announce itself.

```python
# --- flawed_workflow.py: find the five mistakes ---
scaler = StandardScaler().fit(X)              # (1)
X_scaled = scaler.transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

model = Ridge()
best_alpha, best_score = None, float("inf")
for alpha in [0.1, 1, 10, 100]:
    model.set_params(alpha=alpha)
    model.fit(X_train, y_train)
    score = mean_absolute_error(y_test, model.predict(X_test))   # (2)
    if score < best_score:
        best_alpha, best_score = alpha, score

model.set_params(alpha=best_alpha)
model.fit(X_train, np.log(y_train))            # (3)
train_score = mean_absolute_error(y_train, model.predict(X_train))
print(f"MAE: {train_score:.0f}")               # (4), and see (3)

print(f"Best alpha {best_alpha}, expect this MAE on new data: {best_score:.0f}")   # (5)
```

Hint for each, in order: a step that learns from data, run before a split it should come after; a decision made by looking at data it should not have seen yet; a transform applied on one side of an equation and not the other; which set an error is measured against; and what a "best of several" score actually represents.
